# 11 · Build & Manage Taxonomies
**Manage the generated layer: build, inspect, and browse taxonomy files.**

| Step | What happens |
|------|--------------|
| 1 | Check staleness: is a rebuild needed? |
| 2 | Force-rebuild from an Excel definition |
| 3 | Inspect the build manifest |
| 4 | Browse taxonomies interactively (L1 → L2 → description) |

In [ ]:
WORKSPACE = "scopus_content"  # folder under use_cases/ to work in — change to run on another use case
# ── Project root detection ────────────────────────────────────────────────
import sys, os
from pathlib import Path

_root = Path().absolute()
while not (_root / "use_cases").is_dir() and _root.parent != _root:
    _root = _root.parent
_root = _root / "use_cases" / WORKSPACE  # the use-case folder this notebook works in
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f"Project root: {os.getcwd()}")

In [ ]:
import json
import subprocess
import sys
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# ── Settings ─────────────────────────────────────────────────────────────
DEFINITION_FILE = "definitions/scopus_content_pipeline_definition_v2.xlsx"
TAXONOMY_DIR    = "generated/taxonomies"
MANIFEST_PATH   = "generated/build_manifest.json"
CONFIG_PATH     = "generated/pipeline_config.json"

## Step 1 · Staleness check
Compare the definition file's modification time against the manifest timestamp.

### Policy taxonomy harmonisation — CAR Manual v3.37 + Content Coverage Guide (June 2026)

**New policy codes added to `policy_compliance` module:**

| Code | Label | Source |
|------|-------|--------|
| PA10 | Article-in-Press (AiP) eligibility and processing | CAR Manual §5 |
| PA11 | Excluded item types (full CAR §1 exclusion list) | CAR Manual §1 |
| PU2.1–PU2.5 | Customer root cause group | Merged from `policy_understanding` into `policy_compliance` |
| CR1.8 | AiP visibility or missing (contact reason) | CAR Manual §5 |

**Modules changed:**
- `policy_compliance`: expanded with PA10/PA11 policy areas + two new fields (`customer_understanding_issue`, `customer_root_cause_code`)
- `policy_understanding`: **disabled** (`enabled=FALSE`) — its customer dimension fields are now produced by the expanded `policy_compliance` module in a single LLM pass

After updating `definitions/scopus_content_pipeline_definition_v2.xlsx`, run Step 2 (force-rebuild) to regenerate `policy_compliance.json` and `policy_compliance.py`. To regenerate only the compliance module: `rca build build --force policy_compliance`.

### PA12–PA17 additions (June 2026) — first classification run required

**New policy codes added to `policy_compliance` module:**

| Code | Label | Source |
|---|---|---|
| PA12 | Patent indexing and coverage | Content Coverage Guide §2.3 |
| PA13 | Author identity and profile management | Content Coverage Guide §3.3 |
| PA14 | Affiliation profile curation | Content Coverage Guide §3.4 |
| PA15 | Open Access classification | Content Coverage Guide §3.1 |
| PA16 | Errata and record corrections | CCG §3.1 + CAR Manual §6 |
| PA17 | Cited references coverage and linkage | Content Coverage Guide §3.2, §4.5 |

After updating the Excel, rebuild the module:
`rca build build --force policy_compliance`

Then re-run classification (notebook 12) to produce a new `Scopus_Content_Policy_enriched.xlsx` with PA12–PA17 tickets included. The R analysis script (`scopus_effort_master_v1.R`) will automatically handle the new codes — `policy_doc_map` already contains entries for PA12–PA17.


In [ ]:
defn_path    = Path(DEFINITION_FILE)
manifest_path = Path(MANIFEST_PATH)

print(f"Definition file : {defn_path}")
print(f"  exists        : {defn_path.exists()}")
if defn_path.exists():
    defn_mtime = datetime.fromtimestamp(defn_path.stat().st_mtime, tz=timezone.utc)
    print(f"  last modified : {defn_mtime.strftime('%Y-%m-%d %H:%M:%S UTC')}")

print()
print(f"Build manifest  : {manifest_path}")
print(f"  exists        : {manifest_path.exists()}")

rebuild_needed = True
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    gen_at_str = manifest.get("generated_at", "")
    print(f"  generated at  : {gen_at_str}")
    if gen_at_str and defn_path.exists():
        gen_at = datetime.fromisoformat(gen_at_str.replace("Z", "+00:00"))
        rebuild_needed = defn_mtime > gen_at

print()
print(f"Rebuild needed  : {'YES — definition is newer than manifest' if rebuild_needed else 'NO — manifest is current'}")

## Step 2 · Build (force)
Pass `--force-all` to regenerate every file even if unchanged.

In [ ]:
# Remove --force-all to skip unchanged modules (faster incremental build)
result = subprocess.run(
    [sys.executable, "-m", "rca_pipeline.cli", "build",
     "--input", DEFINITION_FILE, "--force-all"],
    capture_output=True, text=True, encoding="utf-8"
)
output = result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout
print(output)
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
else:
    print("Build completed successfully.")

## Step 3 · Inspect the manifest

In [ ]:
import pandas as pd

manifest = json.loads(Path(MANIFEST_PATH).read_text())

print(f"Generator version : {manifest.get('generator_version', '?')}")
print(f"Generated at      : {manifest.get('generated_at', '?')}")
print(f"Input file        : {manifest.get('input_file', '?')}")
print()

modules_meta = manifest.get("modules", {})
if modules_meta:
    rows = []
    for mod, meta in modules_meta.items():
        rows.append({
            "module": mod,
            "taxonomy_file": meta.get("taxonomy_file", ""),
            "classifier_file": meta.get("classifier_file", ""),
            "prefix": meta.get("taxonomy_prefix", ""),
        })
    pd.DataFrame(rows)

In [ ]:
# Summary of generated files on disk
gen_root = Path("generated")

print("Generated taxonomy files:")
for f in sorted((gen_root / "taxonomies").glob("*.json")):
    size = f.stat().st_size
    print(f"  {f.name:<45}  {size:>7,} bytes")

print()
print("Generated classifier modules:")
for f in sorted((gen_root / "modules").glob("*.py")):
    size = f.stat().st_size
    print(f"  {f.name:<45}  {size:>7,} bytes")

## Step 4 · Browse taxonomies
Walk through every L1 node and its L2 children, printing code, label, and description.

In [ ]:
from rca_pipeline.taxonomy import load_taxonomies

taxonomies = load_taxonomies(TAXONOMY_DIR)
print(f"Loaded {len(taxonomies)} taxonomy files: {list(taxonomies.keys())}\n")

for mod_name, tax in sorted(taxonomies.items()):
    l1_nodes = [(k, v) for k, v in sorted(tax.items()) if isinstance(v, dict)]
    print(f"{'=' * 70}")
    print(f"  {mod_name}  —  {len(l1_nodes)} L1 nodes")
    print(f"{'=' * 70}")
    for l1_code, l1_node in l1_nodes:
        l1_label = l1_node.get("label", "")
        l1_desc  = l1_node.get("description", "") or ""
        children = l1_node.get("children", {})
        print(f"\n  [{l1_code}] {l1_label}")
        if l1_desc:
            print(f"    {l1_desc[:80]}{'...' if len(l1_desc) > 80 else ''}")
        for l2_code, l2_node in sorted(children.items()):
            if not isinstance(l2_node, dict):
                continue
            l2_label = l2_node.get("label", "")
            l2_desc  = l2_node.get("description", "") or ""
            print(f"      [{l2_code}] {l2_label}: {l2_desc[:70]}{'...' if len(l2_desc) > 70 else ''}")
    print()

In [ ]:
# Compact summary table: code counts per module
rows = []
for mod_name, tax in taxonomies.items():
    l1_count  = sum(1 for v in tax.values() if isinstance(v, dict))
    l2_count  = sum(
        len(v.get("children", {}))
        for v in tax.values()
        if isinstance(v, dict)
    )
    desc_count = sum(
        1 for v in tax.values()
        if isinstance(v, dict) and v.get("description")
    ) + sum(
        1 for v in tax.values()
        if isinstance(v, dict)
        for c in v.get("children", {}).values()
        if isinstance(c, dict) and c.get("description")
    )
    rows.append(dict(module=mod_name, L1_nodes=l1_count, L2_leaves=l2_count, descriptions=desc_count))

pd.DataFrame(rows).set_index("module")